# APIM ❤️ Self-Hosted Ollama

## Self-Hosted Ollama behind APIM lab

This lab deploys a **CPU-only Ollama** container on **Azure Container Instances** with the `mxbai-embed-large` embedding model, fronted by **Azure API Management**.
The Ollama endpoint is exposed exclusively through APIM so that every call requires a subscription key.

Based on the [Ollama_in_Azure](https://github.com/sorgina13/Ollama_in_Azure) repository.

### Prerequisites

- [Python 3.12 or later version](https://www.python.org/) installed
- [VS Code](https://code.visualstudio.com/) installed with the [Jupyter notebook extension](https://marketplace.visualstudio.com/items?itemName=ms-toolsai.jupyter) enabled
- [Python environment](https://code.visualstudio.com/docs/python/environments#_creating-environments) with the [requirements.txt](../../../requirements.txt) or run `pip install -r requirements.txt` in your terminal
- [An Azure Subscription](https://azure.microsoft.com/free/) with [Contributor](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#contributor) + [RBAC Administrator](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#role-based-access-control-administrator) or [Owner](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#owner) roles
- [Azure CLI](https://learn.microsoft.com/cli/azure/install-azure-cli) installed and [Signed into your Azure subscription](https://learn.microsoft.com/cli/azure/authenticate-azure-cli-interactively)

▶️ Click `Run All` to execute all steps sequentially, or execute them `Step by Step`...

<a id='0'></a>
### 0️⃣ Initialize Variables

In [ ]:
import os, sys, json
sys.path.insert(1, '../../shared')  # add the shared directory to the Python path
import utils

deployment_name = os.path.basename(os.path.dirname(globals()['__vsc_ipynb_file__']))
resource_group_name = f"lab-{deployment_name}"
resource_group_location = "eastus2"

# APIM configuration
apim_sku = 'Basicv2'
apim_subscriptions_config = [{"name": "subscription1", "displayName": "Subscription 1"}]

# Ollama container configuration (uses public ollama/ollama:latest from Docker Hub)
ollama_model = 'mxbai-embed-large'
aci_cpu = 2
aci_memory = 8

utils.print_ok('Notebook initialized')

<a id='1'></a>
### 1️⃣ Verify the Azure CLI and the connected Azure subscription

The following commands ensure that you have the latest version of the Azure CLI and that the Azure CLI is connected to your Azure subscription.

In [ ]:
output = utils.run("az account show", "Retrieved az account", "Failed to get the current az account")

if output.success and output.json_data:
    current_user = output.json_data['user']['name']
    tenant_id = output.json_data['tenantId']
    subscription_id = output.json_data['id']

    utils.print_info(f"Current user: {current_user}")
    utils.print_info(f"Tenant ID: {tenant_id}")
    utils.print_info(f"Subscription ID: {subscription_id}")

<a id='2'></a>
### 2️⃣ Create deployment using 🦾 Bicep

This step creates the resource group and deploys the full infrastructure using [main.bicep](main.bicep):
- **APIM** (Basicv2) as the secure gateway
- **ACI** running the public `ollama/ollama:latest` image from Docker Hub (no ACR build needed)
- **APIM Backend + API** wiring to forward requests to Ollama on ACI

In [ ]:
# Create the resource group
utils.create_resource_group(resource_group_name, resource_group_location)

# Define Bicep parameters
bicep_parameters = {
    "$schema": "https://schema.management.azure.com/schemas/2019-04-01/deploymentParameters.json#",
    "contentVersion": "1.0.0.0",
    "parameters": {
        "apimSku": { "value": apim_sku },
        "apimSubscriptionsConfig": { "value": apim_subscriptions_config },
        "ollamaModel": { "value": ollama_model },
        "aciCpu": { "value": aci_cpu },
        "aciMemoryInGB": { "value": aci_memory },
    }
}

with open('params.json', 'w') as f:
    f.write(json.dumps(bicep_parameters))

utils.print_info('Bicep parameters written to params.json')

# Deploy everything in a single step (uses public ollama/ollama:latest image from Docker Hub)
output = utils.run(
    f"az deployment group create --name {deployment_name} --resource-group {resource_group_name} "
    f"--template-file main.bicep --parameters params.json",
    f"Deployment '{deployment_name}' succeeded",
    f"Deployment '{deployment_name}' failed")

<a id='3'></a>
### 3️⃣ Get the deployment outputs

Retrieve the APIM gateway URL and subscription key needed for testing.

In [ ]:
output = utils.run(
    f"az deployment group show --name {deployment_name} -g {resource_group_name}",
    f"Retrieved deployment: {deployment_name}",
    f"Failed to retrieve deployment: {deployment_name}")

if output.success and output.json_data:
    log_analytics_id = utils.get_deployment_output(output, 'logAnalyticsWorkspaceId', 'Log Analytics Id')
    apim_service_id = utils.get_deployment_output(output, 'apimServiceId', 'APIM Service Id')
    apim_resource_gateway_url = utils.get_deployment_output(output, 'apimResourceGatewayURL', 'APIM API Gateway URL')
    aci_public_ip = utils.get_deployment_output(output, 'aciPublicIP', 'ACI Public IP')
    apim_subscriptions = json.loads(utils.get_deployment_output(output, 'apimSubscriptions').replace("\'", '"'))
    for subscription in apim_subscriptions:
        subscription_name = subscription['name']
        subscription_key = subscription['key']
        utils.print_info(f"Subscription Name: {subscription_name}")
        utils.print_info(f"Subscription Key: ****{subscription_key[-4:]}")
    api_key = apim_subscriptions[0].get('key')

    # Build the APIM base URL for Ollama
    apim_ollama_base_url = f"{apim_resource_gateway_url}/ollama"
    utils.print_info(f"Ollama via APIM: {apim_ollama_base_url}")

<a id='4'></a>
### 4️⃣ Wait for Ollama to be ready

The ACI container needs to download the embedding model on first boot. This step polls until the model is available.

In [ ]:
import time, requests

utils.print_info('Waiting for Ollama model to be ready (this can take a few minutes on first boot)...')

max_attempts = 30
for attempt in range(1, max_attempts + 1):
    try:
        r = requests.get(f'http://{aci_public_ip}:11434/api/tags', timeout=10)
        if r.status_code == 200:
            models = r.json().get('models', [])
            model_names = [m.get('name', '') for m in models]
            if any(ollama_model in name for name in model_names):
                utils.print_ok(f'Ollama is ready with model: {model_names}')
                break
            else:
                utils.print_info(f'Attempt {attempt}/{max_attempts}: Model not yet available. Models: {model_names}')
        else:
            utils.print_info(f'Attempt {attempt}/{max_attempts}: HTTP {r.status_code}')
    except Exception as e:
        utils.print_info(f'Attempt {attempt}/{max_attempts}: {e}')
    time.sleep(20)
else:
    utils.print_error('Ollama did not become ready in time. Check ACI logs with:')
    utils.print_info(f'  az container logs -g {resource_group_name} -n <container-name>')

<a id='test'></a>
### 🧪 Test embeddings through APIM

This test cell reuses the logic from [`olama.py`](https://github.com/sorgina13/Ollama_in_Azure/blob/main/olama.py) but routes all traffic through the APIM gateway using the subscription key.

Tip: Use the [tracing tool](../../../tools/tracing.ipynb) to track the behavior and troubleshoot the [policy](policy.xml).

In [ ]:
import numpy as np
from openai import OpenAI

# ---------- Connect through APIM instead of directly to ACI ----------
client = OpenAI(
    base_url=apim_ollama_base_url,   # APIM gateway URL + /ollama
    api_key="placeholder",            # Required by SDK but not used
    default_headers={"api-key": api_key}  # APIM subscription key
)

# ---------- Sample documents (from olama.py) ----------
documents = [
    "Mixbread AI is a research lab based in Germany.",
    "The mxbai-embed-large model has 335 million parameters.",
    "Ollama allows you to run LLMs and embedding models locally.",
    "Agentic workflows require high-precision retrieval layers."
]

def get_embedding(text, is_query=False):
    prefix = "Represent this sentence for searching relevant passages:\n" if is_query else ""
    response = client.embeddings.create(
        model="mxbai-embed-large",
        input=prefix + text
    )
    return response.data[0].embedding

def cosine_similarity(v1, v2):
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

# ---------- Generate embeddings via APIM → Ollama ----------
print("🧠 Generating embeddings via APIM → Ollama (mxbai-embed-large)...")
doc_vectors = [get_embedding(doc) for doc in documents]

query = "Where is the lab located?"
query_vector = get_embedding(query, is_query=True)

print(f"\n🔍 Query: {query}")
print("-" * 30)

results = []
for i, doc_vec in enumerate(doc_vectors):
    score = cosine_similarity(query_vector, doc_vec)
    results.append((score, documents[i]))

results.sort(key=lambda x: x[0], reverse=True)

for score, text in results:
    print(f"[{score:.4f}] {text}")

utils.print_ok('Embedding test through APIM completed successfully!')

<a id='clean'></a>
### 🗑️ Clean up resources

When you're finished with the lab, you should remove all your deployed resources from Azure to avoid extra charges and keep your Azure subscription uncluttered.
Use the [clean-up-resources notebook](clean-up-resources.ipynb) for that.